# 🧠 Walkthrough : Architecture des Explications LLM Détaillées

Ce notebook récapitule toutes les modifications apportées pour intégrer les explications IA détaillées (en 4 parties) dans l'application SalesTeam AI. Il est conçu pour expliquer les concepts métiers et techniques à un débutant.

## 1️⃣ L'Objectif du Changement

**Avant :**
* La route `/api/recommend` calculait des scores ML et générait pour chaque produit une explication courte de 1 à 2 phrases.
* **Problème :** Si on recommandait 12 produits, on faisait 12 appels à l'API HuggingFace d'un coup. C'était lent, et ça épuisait immédiatement les crédits gratuits de l'API. Dans l'interface, le texte était trop court pour convaincre un client réticent.

**Maintenant :**
* `/api/recommend` reste super rapide. Les petites explications sur les cartes utilisent le LLM, mais on peut les désactiver facilement.
* On a créé une toute nouvelle route `/api/explain-detailed`.
* **Avantage :** L'explication détaillée n'est générée **QUE** lorsque le commercial clique spécifiquement sur un produit. C'est ce qu'on appelle une génération _à la demande_ (on-demand). L'interface a été agrandie en une belle modale redimensionnable avec 4 blocs distincts.

## 2️⃣ Les 3 Piliers de la Nouvelle Architecture

### A. Le Contexte Profond (`src/services/deep_context.py`)
L'IA (le LLM) n'invente rien. Pour qu'elle justifie une recommandation, on doit lui donner des faits têtus. La fonction `retrieve_deep_context()` va fouiller dans les données réelles :
* **Historique :** Extrait les 6 dernières commandes exactes du client (depuis `main_table.csv`).
* **Features ML :** Moyennes, tendances, délais habituels.
* **Score :** Décompose le score mathématique (confiance, boost de timing).
* **Classement :** Regarde qui est le produit juste en-dessous pour comparer.

### B. Le Prompt Engineering (`src/services/explanation.py`)
C'est l'art de parler à l'IA. On injecte toutes les données ci-dessus dans un grand texte (le prompt), suivi de **règles strictes** :
1. Structurer en 4 parties (`Pourquoi ce produit ?`, `Pourquoi cette quantité ?`, etc.).
2. **Zéro hallucination :** Interdiction d'inventer un chiffre qui n'est pas dans le contexte.
3. **Zéro jargon :** Interdiction d'utiliser les mots complexes comme *CV*, *boost*, ou *seuil*.

### C. La Couche API et le Cache (`src/services/recommendation.py`)
Pour ne pas recalculer tout le Machine Learning quand on clique sur une carte, `/api/recommend` sauvegarde son résultat en mémoire (`_last_response_cache`). Quand `/api/explain-detailed` est appelé, il pioche directement dans ce cache.

## 3️⃣ Gestion de l'API HuggingFace et des Crédits

### Qu'est-ce que HuggingFace et le LLM ?
C'est la plateforme qui héberge le modèle d'intelligence artificielle **Llama-3.3-70B-Instruct** (un LLM très puissant de Meta). Quand l'API l'utilise, c'est ce modèle qui rédige le texte comme un humain. Pour lui parler, on utilise un jeton secret (`HF_TOKEN` dans le fichier `.env`).

### Comment savoir si le LLM est actif ?
Quand le LLM est actif, le texte généré est fluide, varie d'une fois à l'autre et relie les concepts. Si la réponse est très rigide et répétitive (ex: *"Ce client a commandé ce produit 11 fois au total..."*), c'est que le LLM est inactif (fallback activé).

### La limite des tokens
L'API gratuite autorise environ 1 000 requêtes par mois. Chaque appel d'explication détaillée consomme environ 600-800 mots (tokens). Si on appelle l'API pour 12 cartes en même temps, le quota est détruit en quelques minutes !

### Comment on a protégé le quota
* **Le Circuit Breaker (`_quota_exhausted`) :** Si HuggingFace renvoie une erreur 402 (Quota Epuisé), l'application l'enregistre (flag à `True`). Les appels suivants n'essaient même plus de contacter HuggingFace pour éviter le spam réseau.
* **Le `skip_llm` :** Lors du calcul massif (l'affichage initial des cartes), on force l'utilisation du fallback déterministe pour réserver 100% des précieux jetons HuggingFace **uniquement** pour l'explication détaillée quand l'utilisateur clique.

## 4️⃣ Le Fallback Déterministe

Que se passe-t-il si l'API HuggingFace est en panne, ou si le quota est épuisé ?

Plutôt que d'afficher une erreur, l'application bascule sur `_rule_based_detailed_explanation()`. C'est une fonction classique (du code Python pur, sans IA) qui prend les mêmes données et génère les 4 paragraphes en utilisant des textes à trous (des *templates* prédéfinis).

C'est moins poétique que l'IA, mais c'est **100% fiable, ultra-rapide et gratuit**.

## 5️⃣ Analyse Critique : Comprendre et Améliorer le LLM

Vous avez cité cet exemple de réponse réelle générée par l'IA :

> *"Le produit M2539E1 WHITE a un niveau de priorité de 2,2422. Il est suivi de la CAMERA C100 qui a un niveau de priorité de 2,053. [...] une augmentation de 0,250"* 

**Le diagnostic de l'expert :**
L'IA a techniquement obéi : elle n'a pas utilisé le mot "score" ou "trend" qui étaient interdits. Cependant, comme on lui a passé les valeurs mathématiques exactes du Machine Learning, elle les a bêtement répétées. Résultat : des décimales incompréhensibles pour un commercial non-connaisseur (`2,2422` ou `0,250`).

**Comment corriger cela dans une prochaine étape ?**
Il faut agir sur le `_build_detailed_prompt()` dans `explanation.py`. Actuellement, on dit à l'IA : "*Ne dis pas de jargon*". Il faut aller plus loin et filtrer les données **avant** de les lui donner :
1. **Masquer les décimales :** Ne jamais envoyer `2.2422` dans le prompt. Le backend Python doit l'arrondir (`round(score, 1)` -> `2.2`) ou même ne donner que le rang (1er, 2ème).
2. **Traduire la tendance en mots :** Plutôt que d'envoyer `0.250` pour la tendance, le code Python devrait traduire ça en texte pour le prompt : `"Tendance : en légère hausse"`. On supprime complètement la valeur chiffrée de la portée de l'IA.
3. **Règle absolue renforcée :** Ajouter dans le prompt système : *"Ne cite AUCUN chiffre à virgule complexe. Si tu dois comparer deux produits, dis simplement que l'un est largement prioritaire par rapport à l'autre sans citer la valeur exacte du niveau de priorité."*

C'est la beauté du Prompt Engineering : ce que l'IA ne voit pas, elle ne peut pas s'y accrocher. En filtrant les données brutes en amont, on contrôle parfaitement son niveau de vulgarisation métier !